### Understand the dataset better

A: traits x characters (464 x 2000)

Notes about the SVD:

- A * V : traits x characters
- U.T * A : archertype space (basis for characters) -- 464 x 2000 (traits x PC of archetypes)

Notes:

1. We could just use archetype_df as input to the clustering/classification model ... but there are a lot of 0s
2. We could normalize this - does this deal with stories having differing numbers of primary characters?

In [35]:
import numpy as np
import pandas as pd

In [36]:
# show all columns
pd.set_option('display.max_columns', None)

Load & Preprocess Data

In [37]:
df = pd.read_csv('../prepared_data/characters_with_imdb.csv')

In [39]:
# For each story_name, group characters by their primary archetype frequency,
# while keeping genre columns that were added before this groupby.
archetype_freqs = (
    df.groupby(['story_name', 'genre', 'is_comedy'])['primary_archetype']
    .value_counts()
    .unstack(fill_value=0)
)

# Convert grouped frequencies to dataframe and keep story_name as index
archetype_freqs_df = archetype_freqs.reset_index().set_index('story_name')

# Archetype feature columns (exclude metadata columns)
archetype_cols = [c for c in archetype_freqs_df.columns if c not in ['genre', 'is_comedy']]

archetype_freqs_df.head()

primary_archetype,genre,is_comedy,Adventurer,Adventurer-Angel,Adventurer-Angel-Diva,Adventurer-Angel-Hero,Adventurer-Angel-Outcast,Adventurer-Brute,Adventurer-Brute-Demon,Adventurer-Brute-Hero,Adventurer-Demon,Adventurer-Demon-Diva,Adventurer-Demon-Fool,Adventurer-Demon-Hero,Adventurer-Demon-Lone Wolf,Adventurer-Demon-Outcast,Adventurer-Diva,Adventurer-Diva-Angel,Adventurer-Diva-Demon,Adventurer-Diva-Hero,Adventurer-Fool-Demon,Adventurer-Fool-Outcast,Adventurer-Geek-Outcast,Adventurer-Hero,Adventurer-Hero-Angel,Adventurer-Hero-Brute,Adventurer-Hero-Demon,Adventurer-Hero-Diva,Adventurer-Hero-Geek,Adventurer-Hero-Outcast,Adventurer-Lone Wolf-Demon,Adventurer-Lone Wolf-Hero,Adventurer-Outcast,Adventurer-Outcast-Angel,Adventurer-Outcast-Demon,Adventurer-Outcast-Diva,Adventurer-Outcast-Hero,Adventurer-Outcast-Lone Wolf,Adventurer-Sophisticate-Hero,Angel,Angel-Adventurer,Angel-Adventurer-Diva,Angel-Adventurer-Hero,Angel-Adventurer-Outcast,Angel-Brute,Angel-Brute-Adventurer,Angel-Brute-Hero,Angel-Diva-Adventurer,Angel-Diva-Hero,Angel-Diva-Outcast,Angel-Diva-Traditionalist,Angel-Fool,Angel-Geek-Adventurer,Angel-Geek-Hero,Angel-Geek-Outcast,Angel-Hero,Angel-Hero-Adventurer,Angel-Hero-Diva,Angel-Hero-Lone Wolf,Angel-Hero-Traditionalist,Angel-Lone Wolf-Hero,Angel-Outcast,Angel-Outcast-Adventurer,Angel-Outcast-Diva,Angel-Outcast-Hero,Angel-Outcast-Traditionalist,Angel-Traditionalist-Hero,Angel-Traditionalist-Outcast,Brute-Adventurer,Brute-Adventurer-Angel,Brute-Adventurer-Demon,Brute-Adventurer-Diva,Brute-Adventurer-Fool,Brute-Adventurer-Hero,Brute-Angel,Brute-Angel-Adventurer,Brute-Angel-Hero,Brute-Angel-Lone Wolf,Brute-Demon,Brute-Demon-Adventurer,Brute-Demon-Diva,Brute-Demon-Hero,Brute-Demon-Outcast,Brute-Demon-Traditionalist,Brute-Diva,Brute-Diva-Adventurer,Brute-Diva-Angel,Brute-Diva-Demon,Brute-Diva-Hero,Brute-Fool-Adventurer,Brute-Hero,Brute-Hero-Adventurer,Brute-Hero-Angel,Brute-Hero-Demon,Brute-Hero-Diva,Brute-Hero-Outcast,Brute-Hero-Traditionalist,Brute-Lone Wolf-Hero,Brute-Lone Wolf-Outcast,Brute-Outcast,Brute-Outcast-Adventurer,Brute-Outcast-Angel,Brute-Outcast-Demon,Brute-Outcast-Diva,Brute-Outcast-Hero,Brute-Traditionalist,Brute-Traditionalist-Demon,Brute-Traditionalist-Hero,Brute-Traditionalist-Outcast,Demon,Demon-Adventurer,Demon-Adventurer-Brute,Demon-Adventurer-Diva,Demon-Adventurer-Hero,Demon-Adventurer-Lone Wolf,Demon-Adventurer-Outcast,Demon-Brute-Adventurer,Demon-Brute-Diva,Demon-Brute-Hero,Demon-Brute-Outcast,Demon-Diva,Demon-Diva-Adventurer,Demon-Diva-Hero,Demon-Diva-Traditionalist,Demon-Fool-Diva,Demon-Fool-Outcast,Demon-Geek-Adventurer,Demon-Geek-Hero,Demon-Hero,Demon-Hero-Adventurer,Demon-Hero-Diva,Demon-Hero-Geek,Demon-Hero-Lone Wolf,Demon-Hero-Outcast,Demon-Hero-Traditionalist,Demon-Lone Wolf-Adventurer,Demon-Lone Wolf-Hero,Demon-Lone Wolf-Outcast,Demon-Outcast,Demon-Outcast-Adventurer,Demon-Outcast-Fool,Demon-Outcast-Hero,Demon-Outcast-Lone Wolf,Demon-Outcast-Traditionalist,Demon-Sophisticate-Hero,Demon-Traditionalist-Diva,Demon-Traditionalist-Hero,Diva,Diva-Adventurer,Diva-Adventurer-Angel,Diva-Adventurer-Demon,Diva-Adventurer-Fool,Diva-Adventurer-Hero,Diva-Adventurer-Outcast,Diva-Angel,Diva-Angel-Adventurer,Diva-Angel-Brute,Diva-Angel-Hero,Diva-Angel-Outcast,Diva-Demon,Diva-Demon-Adventurer,Diva-Demon-Brute,Diva-Demon-Fool,Diva-Demon-Hero,Diva-Demon-Traditionalist,Diva-Fool-Adventurer,Diva-Fool-Outcast,Diva-Geek-Adventurer,Diva-Geek-Demon,Diva-Geek-Hero,Diva-Geek-Outcast,Diva-Hero,Diva-Hero-Adventurer,Diva-Hero-Angel,Diva-Hero-Brute,Diva-Hero-Demon,Diva-Hero-Outcast,Diva-Hero-Traditionalist,Diva-Outcast,Diva-Outcast-Adventurer,Diva-Outcast-Angel,Diva-Outcast-Hero,Diva-Outcast-Traditionalist,Diva-Sophisticate-Adventurer,Diva-Sophisticate-Hero,Diva-Traditionalist,Diva-Traditionalist-Angel,Diva-Traditionalist-Demon,Diva-Traditionalist-Hero,Diva-Traditionalist-Outcast,Fool-Adventurer,Fool-Adventurer-Demon,Fool-Adventurer-Diva,Fool-Adventurer-Outcast,Fool-Angel-Outcast,Fool-Brute-Adventurer,Fool-Demon-Adventurer

In [40]:
# Normalize archetype frequencies by total characters in each story
archetype_freqs_df_norm = archetype_freqs_df[archetype_cols].div(
    archetype_freqs_df[archetype_cols].sum(axis=1),
    axis=0
)

# Keep labels alongside normalized features
archetype_freqs_df_norm = archetype_freqs_df[['genre', 'is_comedy']].join(archetype_freqs_df_norm)
archetype_freqs_df_norm.head()

primary_archetype,genre,is_comedy,Adventurer,Adventurer-Angel,Adventurer-Angel-Diva,Adventurer-Angel-Hero,Adventurer-Angel-Outcast,Adventurer-Brute,Adventurer-Brute-Demon,Adventurer-Brute-Hero,Adventurer-Demon,Adventurer-Demon-Diva,Adventurer-Demon-Fool,Adventurer-Demon-Hero,Adventurer-Demon-Lone Wolf,Adventurer-Demon-Outcast,Adventurer-Diva,Adventurer-Diva-Angel,Adventurer-Diva-Demon,Adventurer-Diva-Hero,Adventurer-Fool-Demon,Adventurer-Fool-Outcast,Adventurer-Geek-Outcast,Adventurer-Hero,Adventurer-Hero-Angel,Adventurer-Hero-Brute,Adventurer-Hero-Demon,Adventurer-Hero-Diva,Adventurer-Hero-Geek,Adventurer-Hero-Outcast,Adventurer-Lone Wolf-Demon,Adventurer-Lone Wolf-Hero,Adventurer-Outcast,Adventurer-Outcast-Angel,Adventurer-Outcast-Demon,Adventurer-Outcast-Diva,Adventurer-Outcast-Hero,Adventurer-Outcast-Lone Wolf,Adventurer-Sophisticate-Hero,Angel,Angel-Adventurer,Angel-Adventurer-Diva,Angel-Adventurer-Hero,Angel-Adventurer-Outcast,Angel-Brute,Angel-Brute-Adventurer,Angel-Brute-Hero,Angel-Diva-Adventurer,Angel-Diva-Hero,Angel-Diva-Outcast,Angel-Diva-Traditionalist,Angel-Fool,Angel-Geek-Adventurer,Angel-Geek-Hero,Angel-Geek-Outcast,Angel-Hero,Angel-Hero-Adventurer,Angel-Hero-Diva,Angel-Hero-Lone Wolf,Angel-Hero-Traditionalist,Angel-Lone Wolf-Hero,Angel-Outcast,Angel-Outcast-Adventurer,Angel-Outcast-Diva,Angel-Outcast-Hero,Angel-Outcast-Traditionalist,Angel-Traditionalist-Hero,Angel-Traditionalist-Outcast,Brute-Adventurer,Brute-Adventurer-Angel,Brute-Adventurer-Demon,Brute-Adventurer-Diva,Brute-Adventurer-Fool,Brute-Adventurer-Hero,Brute-Angel,Brute-Angel-Adventurer,Brute-Angel-Hero,Brute-Angel-Lone Wolf,Brute-Demon,Brute-Demon-Adventurer,Brute-Demon-Diva,Brute-Demon-Hero,Brute-Demon-Outcast,Brute-Demon-Traditionalist,Brute-Diva,Brute-Diva-Adventurer,Brute-Diva-Angel,Brute-Diva-Demon,Brute-Diva-Hero,Brute-Fool-Adventurer,Brute-Hero,Brute-Hero-Adventurer,Brute-Hero-Angel,Brute-Hero-Demon,Brute-Hero-Diva,Brute-Hero-Outcast,Brute-Hero-Traditionalist,Brute-Lone Wolf-Hero,Brute-Lone Wolf-Outcast,Brute-Outcast,Brute-Outcast-Adventurer,Brute-Outcast-Angel,Brute-Outcast-Demon,Brute-Outcast-Diva,Brute-Outcast-Hero,Brute-Traditionalist,Brute-Traditionalist-Demon,Brute-Traditionalist-Hero,Brute-Traditionalist-Outcast,Demon,Demon-Adventurer,Demon-Adventurer-Brute,Demon-Adventurer-Diva,Demon-Adventurer-Hero,Demon-Adventurer-Lone Wolf,Demon-Adventurer-Outcast,Demon-Brute-Adventurer,Demon-Brute-Diva,Demon-Brute-Hero,Demon-Brute-Outcast,Demon-Diva,Demon-Diva-Adventurer,Demon-Diva-Hero,Demon-Diva-Traditionalist,Demon-Fool-Diva,Demon-Fool-Outcast,Demon-Geek-Adventurer,Demon-Geek-Hero,Demon-Hero,Demon-Hero-Adventurer,Demon-Hero-Diva,Demon-Hero-Geek,Demon-Hero-Lone Wolf,Demon-Hero-Outcast,Demon-Hero-Traditionalist,Demon-Lone Wolf-Adventurer,Demon-Lone Wolf-Hero,Demon-Lone Wolf-Outcast,Demon-Outcast,Demon-Outcast-Adventurer,Demon-Outcast-Fool,Demon-Outcast-Hero,Demon-Outcast-Lone Wolf,Demon-Outcast-Traditionalist,Demon-Sophisticate-Hero,Demon-Traditionalist-Diva,Demon-Traditionalist-Hero,Diva,Diva-Adventurer,Diva-Adventurer-Angel,Diva-Adventurer-Demon,Diva-Adventurer-Fool,Diva-Adventurer-Hero,Diva-Adventurer-Outcast,Diva-Angel,Diva-Angel-Adventurer,Diva-Angel-Brute,Diva-Angel-Hero,Diva-Angel-Outcast,Diva-Demon,Diva-Demon-Adventurer,Diva-Demon-Brute,Diva-Demon-Fool,Diva-Demon-Hero,Diva-Demon-Traditionalist,Diva-Fool-Adventurer,Diva-Fool-Outcast,Diva-Geek-Adventurer,Diva-Geek-Demon,Diva-Geek-Hero,Diva-Geek-Outcast,Diva-Hero,Diva-Hero-Adventurer,Diva-Hero-Angel,Diva-Hero-Brute,Diva-Hero-Demon,Diva-Hero-Outcast,Diva-Hero-Traditionalist,Diva-Outcast,Diva-Outcast-Adventurer,Diva-Outcast-Angel,Diva-Outcast-Hero,Diva-Outcast-Traditionalist,Diva-Sophisticate-Adventurer,Diva-Sophisticate-Hero,Diva-Traditionalist,Diva-Traditionalist-Angel,Diva-Traditionalist-Demon,Diva-Traditionalist-Hero,Diva-Traditionalist-Outcast,Fool-Adventurer,Fool-Adventurer-Demon,Fool-Adventurer-Diva,Fool-Adventurer-Outcast,Fool-Angel-Outcast,Fool-Brute-Adventurer,Fool-Demon-Adventurer

In [44]:
# Value counts of genres in archetype_freqs_df
archetype_freqs_df['genre'].value_counts()

genre
drama     188
comedy    139
Name: count, dtype: int64

In [41]:
# What are the most common primary archetypes across all stories?
archetype_freqs_df[archetype_cols].sum(axis=0).sort_values(ascending=False).head(20)

primary_archetype
Hero                     179
Angel-Hero                74
Adventurer                61
Hero-Demon                52
Demon                     43
Demon-Hero                39
Traditionalist-Hero       36
Hero-Angel                33
Adventurer-Hero           28
Angel                     27
Angel-Adventurer          25
Lone Wolf-Hero            20
Hero-Adventurer           19
Demon-Adventurer          18
Diva-Angel-Hero           17
Diva-Adventurer           16
Hero-Angel-Adventurer     15
Hero-Demon-Adventurer     14
Diva-Demon                14
Brute-Angel-Hero          14
dtype: int64

In [31]:
# What are the most common primary archetypes across all stories, normalized by the total number of characters in each story?
# Is it the same as the non-normalized version? Not exactly.
archetype_freqs_df_norm[archetype_cols].sum(axis=0).sort_values(ascending=False).head(20)

Hero                     30.233333
Adventurer               14.066667
Angel-Hero               13.633333
Hero-Demon                8.900000
Demon-Hero                8.066667
Demon                     6.766667
Traditionalist-Hero       6.266667
Adventurer-Hero           6.000000
Hero-Angel                5.566667
Angel                     5.133333
Lone Wolf-Hero            4.733333
Angel-Adventurer          4.466667
Demon-Adventurer          3.733333
Diva-Angel-Hero           3.733333
Hero-Angel-Adventurer     3.700000
Hero-Adventurer           3.400000
Adventurer-Angel-Hero     2.766667
Outcast-Angel-Hero        2.666667
Angel-Hero-Adventurer     2.666667
Diva-Hero                 2.633333
dtype: float64

In [20]:
# Filter index to The Office
archetype_freqs_df_office = archetype_freqs_df[archetype_freqs_df.index.str.contains('The Office')]
archetype_freqs_df_office.head()

,Adventurer,Adventurer-Angel,Adventurer-Angel-Diva,Adventurer-Angel-Hero,Adventurer-Angel-Outcast,Adventurer-Brute,Adventurer-Brute-Demon,Adventurer-Brute-Hero,Adventurer-Demon,Adventurer-Demon-Diva,Adventurer-Demon-Fool,Adventurer-Demon-Hero,Adventurer-Demon-Lone Wolf,Adventurer-Demon-Outcast,Adventurer-Diva,Adventurer-Diva-Angel,Adventurer-Diva-Demon,Adventurer-Diva-Hero,Adventurer-Fool-Demon,Adventurer-Fool-Outcast,Adventurer-Geek-Outcast,Adventurer-Hero,Adventurer-Hero-Angel,Adventurer-Hero-Brute,Adventurer-Hero-Demon,Adventurer-Hero-Diva,Adventurer-Hero-Geek,Adventurer-Hero-Outcast,Adventurer-Lone Wolf-Demon,Adventurer-Lone Wolf-Hero,Adventurer-Outcast,Adventurer-Outcast-Angel,Adventurer-Outcast-Demon,Adventurer-Outcast-Diva,Adventurer-Outcast-Hero,Adventurer-Outcast-Lone Wolf,Adventurer-Sophisticate-Hero,Angel,Angel-Adventurer,Angel-Adventurer-Diva,Angel-Adventurer-Hero,Angel-Adventurer-Outcast,Angel-Brute,Angel-Brute-Adventurer,Angel-Brute-Hero,Angel-Diva-Adventurer,Angel-Diva-Hero,Angel-Diva-Outcast,Angel-Diva-Traditionalist,Angel-Fool,Angel-Geek-Adventurer,Angel-Geek-Hero,Angel-Geek-Outcast,Angel-Hero,Angel-Hero-Adventurer,Angel-Hero-Diva,Angel-Hero-Lone Wolf,Angel-Hero-Traditionalist,Angel-Lone Wolf-Hero,Angel-Outcast,Angel-Outcast-Adventurer,Angel-Outcast-Diva,Angel-Outcast-Hero,Angel-Outcast-Traditionalist,Angel-Traditionalist-Hero,Angel-Traditionalist-Outcast,Brute-Adventurer,Brute-Adventurer-Angel,Brute-Adventurer-Demon,Brute-Adventurer-Diva,Brute-Adventurer-Fool,Brute-Adventurer-Hero,Brute-Angel,Brute-Angel-Adventurer,Brute-Angel-Hero,Brute-Angel-Lone Wolf,Brute-Demon,Brute-Demon-Adventurer,Brute-Demon-Diva,Brute-Demon-Hero,Brute-Demon-Outcast,Brute-Demon-Traditionalist,Brute-Diva,Brute-Diva-Adventurer,Brute-Diva-Angel,Brute-Diva-Demon,Brute-Diva-Hero,Brute-Fool-Adventurer,Brute-Hero,Brute-Hero-Adventurer,Brute-Hero-Angel,Brute-Hero-Demon,Brute-Hero-Diva,Brute-Hero-Outcast,Brute-Hero-Traditionalist,Brute-Lone Wolf-Hero,Brute-Lone Wolf-Outcast,Brute-Outcast,Brute-Outcast-Adventurer,Brute-Outcast-Angel,Brute-Outcast-Demon,Brute-Outcast-Diva,Brute-Outcast-Hero,Brute-Traditionalist,Brute-Traditionalist-Demon,Brute-Traditionalist-Hero,Brute-Traditionalist-Outcast,Demon,Demon-Adventurer,Demon-Adventurer-Brute,Demon-Adventurer-Diva,Demon-Adventurer-Hero,Demon-Adventurer-Lone Wolf,Demon-Adventurer-Outcast,Demon-Brute-Adventurer,Demon-Brute-Diva,Demon-Brute-Hero,Demon-Brute-Outcast,Demon-Diva,Demon-Diva-Adventurer,Demon-Diva-Hero,Demon-Diva-Traditionalist,Demon-Fool-Diva,Demon-Fool-Outcast,Demon-Geek-Adventurer,Demon-Geek-Hero,Demon-Hero,Demon-Hero-Adventurer,Demon-Hero-Brute,Demon-Hero-Diva,Demon-Hero-Geek,Demon-Hero-Lone Wolf,Demon-Hero-Outcast,Demon-Hero-Traditionalist,Demon-Lone Wolf-Adventurer,Demon-Lone Wolf-Hero,Demon-Lone Wolf-Outcast,Demon-Outcast,Demon-Outcast-Adventurer,Demon-Outcast-Fool,Demon-Outcast-Hero,Demon-Outcast-Lone Wolf,Demon-Outcast-Traditionalist,Demon-Sophisticate-Hero,Demon-Traditionalist-Diva,Demon-Traditionalist-Hero,Diva,Diva-Adventurer,Diva-Adventurer-Angel,Diva-Adventurer-Demon,Diva-Adventurer-Fool,Diva-Adventurer-Hero,Diva-Adventurer-Outcast,Diva-Angel,Diva-Angel-Adventurer,Diva-Angel-Brute,Diva-Angel-Hero,Diva-Angel-Outcast,Diva-Demon,Diva-Demon-Adventurer,Diva-Demon-Brute,Diva-Demon-Fool,Diva-Demon-Hero,Diva-Demon-Traditionalist,Diva-Fool-Adventurer,Diva-Fool-Outcast,Diva-Geek-Adventurer,Diva-Geek-Demon,Diva-Geek-Hero,Diva-Geek-Outcast,Diva-Hero,Diva-Hero-Adventurer,Diva-Hero-Angel,Diva-Hero-Brute,Diva-Hero-Demon,Diva-Hero-Outcast,Diva-Hero-Traditionalist,Diva-Outcast,Diva-Outcast-Adventurer,Diva-Outcast-Angel,Diva-Outcast-Hero,Diva-Outcast-Traditionalist,Diva-Sophisticate-Adventurer,Diva-Sophisticate-Hero,Diva-Traditionalist,Diva-Traditionalist-Angel,Diva-Traditionalist-Demon,Diva-Traditionalist-Hero,Diva-Traditionalist-Outcast,Fool-Adventurer,Fool-Adventurer-Demon,Fool-Adventurer-Diva,Fool-Adventurer-Outcast,Fool-Angel-Outcast,Fool-Brute-Adventurer,Fool-Demon-Adventurer,Fool-Demon-Outc